# 03. 피처 생성 — 개별 패널 CSV 산출

**목적**: `02_data_collection.ipynb` 에서 수집한 캐시(.pkl)와 FF 팩터를 결합해,
110개 종목 각각에 대한 **날짜 인덱스 + 약 23개 컬럼** 의 패널 CSV 를 생성한다.

**파이프라인 (종목당)**
1. `data/cache/{ticker}.pkl` 로드 → 가격 컬럼 정규화
2. `adj_close` 기반 일별 수익률 (log/simple) 계산
3. 시가총액: `adj_close × shares_outstanding` (현재 주식수 기준 근사치)
4. `data/ff_factors.csv` Left-join (날짜 키)
5. 모멘텀 5종 (`mom_1m`, `mom_3m`, `mom_6m`, `mom_12m`, `mom_12m_skip_1m`)
6. 변동성 3종 (`vol_20d_ann`, `vol_60d_ann`, `vol_252d_ann`)
7. 컬럼 순서 고정 + `data/panels/{ticker}.csv` 저장

**산출물**: `data/panels/{ticker}.csv` (110개, 각 약 4,000행 × 23컬럼)

---
## Section 0. 설정 및 함수 정의

In [5]:
import os
import pickle
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

# ── 경로 설정 ────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR     = PROJECT_DIR / 'data'
CACHE_DIR    = DATA_DIR / 'cache'
PANELS_DIR   = DATA_DIR / 'panels'

for d in [DATA_DIR, CACHE_DIR, PANELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def setup_korean_font():
    """matplotlib 한글 폰트 설정 — OS별 분기."""
    if platform.system() == 'Windows':
        plt.rcParams['font.family'] = 'Malgun Gothic'
    elif platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'AppleGothic'
    else:
        try:
            import koreanize_matplotlib  # noqa: F401
        except ImportError:
            pass
    plt.rcParams['axes.unicode_minus'] = False


setup_korean_font()


def compute_returns(adj_close: pd.Series) -> pd.DataFrame:
    """adj_close 시계열로부터 일별 수익률(로그/단순) 계산.

    로그 수익률은 분산 추정에, 단순 수익률은 모멘텀 계산에 주로 사용한다.
    """
    log_ret    = np.log(adj_close / adj_close.shift(1))
    simple_ret = adj_close.pct_change()
    return pd.DataFrame({
        'log_return_1d':    log_ret,
        'simple_return_1d': simple_ret,
    })


def compute_momentum(simple_ret: pd.Series) -> pd.DataFrame:
    """단순 수익률로부터 모멘텀 5종 계산.

    윈도우(영업일 기준):
      - mom_1m:  21일 (≈1개월)
      - mom_3m:  63일 (≈3개월)
      - mom_6m:  126일 (≈6개월)
      - mom_12m: 252일 (≈12개월)
      - mom_12m_skip_1m: 12개월 모멘텀에서 최근 1개월을 제외 (학술 표준)
        계산식: 롤링 252일 누적수익률 ÷ 롤링 21일 누적수익률 - 1
    """
    r = 1 + simple_ret  # 1 + r(t)
    prod_252 = r.rolling(252).apply(np.prod, raw=True)
    prod_21  = r.rolling(21).apply(np.prod, raw=True)

    return pd.DataFrame({
        'mom_1m':          r.rolling(21).apply(np.prod, raw=True)  - 1,
        'mom_3m':          r.rolling(63).apply(np.prod, raw=True)  - 1,
        'mom_6m':          r.rolling(126).apply(np.prod, raw=True) - 1,
        'mom_12m':         prod_252 - 1,
        'mom_12m_skip_1m': prod_252 / prod_21 - 1,  # 최근 1개월 제외한 12개월 모멘텀
    })


def compute_volatility(log_ret: pd.Series) -> pd.DataFrame:
    """로그 수익률로부터 연환산 변동성 3종 계산.

    연환산 변동성 = 롤링 표본 표준편차 × √252
    로그 수익률이 정규분포에 더 근접하므로 변동성 추정에 적합하다.
    """
    ann = np.sqrt(252)
    return pd.DataFrame({
        'vol_20d_ann':  log_ret.rolling(20).std()  * ann,
        'vol_60d_ann':  log_ret.rolling(60).std()  * ann,
        'vol_252d_ann': log_ret.rolling(252).std() * ann,
    })


def build_panel(
    ticker: str,
    gics_sector: str,
    price_df: pd.DataFrame,
    ff_df: pd.DataFrame,
    shares: float = None,
) -> pd.DataFrame:
    """가격 DataFrame + FF 팩터를 결합해 최종 패널 DataFrame 생성.

    출력 컬럼 순서(23개):
      ticker, gics_sector,
      open, high, low, close, adj_close, volume, dividends, split_ratio,
      log_return_1d, simple_return_1d, market_cap,
      mkt_rf, smb, hml, rmw, cma, rf, mom_factor,
      mom_1m, mom_3m, mom_6m, mom_12m, mom_12m_skip_1m,
      vol_20d_ann, vol_60d_ann, vol_252d_ann

    Parameters
    ----------
    shares : float, optional
        현재 주식수. 없으면 market_cap = NaN.
    """
    # (1) 컬럼명 소문자·공백→언더스코어 정규화
    df = price_df.copy()
    df.columns = (
        df.columns.str.strip()
                  .str.lower()
                  .str.replace(' ', '_', regex=False)
    )
    # (2) 표준 컬럼명으로 매핑
    rename = {
        'adj_close': 'adj_close',      # 이미 정규화된 경우
        'adjclose':  'adj_close',
        'stock_splits': 'split_ratio',
    }
    df = df.rename(columns=rename)

    # 필수 컬럼 존재 확인
    if 'adj_close' not in df.columns and 'close' in df.columns:
        # auto_adjust=True 로 받았거나, 컬럼이 다른 경우 fallback
        df['adj_close'] = df['close']

    if 'split_ratio' not in df.columns:
        df['split_ratio'] = 0.0  # 분할 없음
    if 'dividends' not in df.columns:
        df['dividends'] = 0.0

    # (3) 수익률
    ret = compute_returns(df['adj_close'])

    # (4) 시가총액 (현재 주식수 × adj_close, 시점별 근사)
    df['market_cap'] = (df['adj_close'] * shares) if shares else np.nan

    # (5) FF 팩터 Left-join (날짜 기준)
    ff_cols = ['mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'rf', 'mom_factor']
    existing_ff_cols = [c for c in ff_cols if c in ff_df.columns]
    df = df.join(ff_df[existing_ff_cols], how='left')

    # (6) 피처 계산
    mom = compute_momentum(ret['simple_return_1d'])
    vol = compute_volatility(ret['log_return_1d'])

    df = df.join(ret, how='left')
    df = df.join(mom, how='left')
    df = df.join(vol, how='left')

    # (7) 식별 컬럼 + 컬럼 순서 고정
    df['ticker']      = ticker
    df['gics_sector'] = gics_sector

    ordered_cols = [
        'ticker', 'gics_sector',
        'open', 'high', 'low', 'close', 'adj_close', 'volume',
        'dividends', 'split_ratio',
        'log_return_1d', 'simple_return_1d', 'market_cap',
        'mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'rf', 'mom_factor',
        'mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'mom_12m_skip_1m',
        'vol_20d_ann', 'vol_60d_ann', 'vol_252d_ann',
    ]
    # 실제 존재하는 컬럼만 선택 (없는 컬럼은 NaN 으로 채움)
    for col in ordered_cols:
        if col not in df.columns:
            df[col] = np.nan

    return df[ordered_cols]


print(f'프로젝트 경로 : {PROJECT_DIR}')
print(f'캐시 경로     : {CACHE_DIR}')
print(f'패널 출력     : {PANELS_DIR}')
print('함수 정의 완료')

프로젝트 경로 : c:\Users\gorhk\최종 프로젝트\finance_project\김재천\black_litterman
캐시 경로     : c:\Users\gorhk\최종 프로젝트\finance_project\김재천\black_litterman\data\cache
패널 출력     : c:\Users\gorhk\최종 프로젝트\finance_project\김재천\black_litterman\data\panels
함수 정의 완료


---
## Section 1. 패널 CSV 생성 파이프라인

**롤링 계산 NaN 규칙**
- 모멘텀: 윈도우 미만 구간(예: 252일 미만)은 NaN → 0 채우기 금지
- 변동성: 윈도우 미만 구간은 NaN → 0 채우기 금지
- FF 팩터: 최신 1~2개월 데이터 지연 → NaN 유지

**시가총액 근사**
- yfinance `info['sharesOutstanding']` × `adj_close` (현재 주식수 기준, 과거 희석 반영 안 됨)
- 주식수 정보가 없으면 `market_cap = NaN`

In [6]:
# universe.csv + ff_factors.csv 로드
universe_path = DATA_DIR / 'universe.csv'
ff_path       = DATA_DIR / 'ff_factors.csv'

assert universe_path.exists(), (
    f'universe.csv 없음 → 01_universe_selection.ipynb 를 먼저 실행하세요'
)
assert ff_path.exists(), (
    f'ff_factors.csv 없음 → 02_data_collection.ipynb 를 먼저 실행하세요'
)

df_universe = pd.read_csv(universe_path)
df_ff = pd.read_csv(ff_path, index_col='date', parse_dates=True)

print(f'유니버스 종목 수 : {len(df_universe)}')
print(f'FF 팩터 행 수   : {len(df_ff)}')
print(f'FF 팩터 컬럼    : {df_ff.columns.tolist()}')

# 섹터 매핑 딕셔너리 (ticker → gics_sector)
sector_map = dict(zip(df_universe['ticker'], df_universe['gics_sector']))

유니버스 종목 수 : 110
FF 팩터 행 수   : 4024
FF 팩터 컬럼    : ['mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'rf', 'mom_factor']


In [7]:
# 패널 CSV 생성 루프 (110 종목)
ok_list   = []
fail_list = []

tickers = df_universe['ticker'].tolist()

for i, ticker in enumerate(tickers, 1):
    panel_path = PANELS_DIR / f'{ticker}.csv'
    cache_path = CACHE_DIR / f'{ticker}.pkl'

    # 이미 패널 CSV 있으면 스킵
    if panel_path.exists():
        ok_list.append(ticker)
        if i % 30 == 0 or i == len(tickers):
            print(f'  [{i:3d}/{len(tickers)}] (패널 스킵 중... {ticker})')
        continue

    # 캐시 없으면 스킵 (02 노트북에서 수집 실패한 종목)
    if not cache_path.exists():
        fail_list.append(ticker)
        print(f'  [{i:3d}/{len(tickers)}] {ticker:8s} → 캐시 없음 SKIP')
        continue

    # 가격 데이터 로드
    with open(cache_path, 'rb') as f:
        df_price = pickle.load(f)

    # 현재 주식수 조회 (시가총액 근사용, 없어도 진행)
    shares = None
    try:
        info = yf.Ticker(ticker).info
        shares = info.get('sharesOutstanding') or info.get('impliedSharesOutstanding')
        if shares:
            shares = float(shares)
    except Exception:
        pass

    # 패널 빌드
    try:
        gics_sector = sector_map.get(ticker, 'Unknown')
        df_panel = build_panel(ticker, gics_sector, df_price, df_ff, shares)
        df_panel.to_csv(panel_path, index=True, encoding='utf-8-sig')
        ok_list.append(ticker)
        if i % 10 == 0 or i == len(tickers):
            print(f'  [{i:3d}/{len(tickers)}] {ticker:8s} → OK '
                  f'({len(df_panel)}행, NaN market_cap={df_panel["market_cap"].isna().all()})')
    except Exception as e:
        fail_list.append(ticker)
        print(f'  [{i:3d}/{len(tickers)}] {ticker:8s} → FAIL: {e}')

print(f'\n완료: 성공 {len(ok_list)}개 / 실패 {len(fail_list)}개')
if fail_list:
    print(f'실패 종목: {fail_list}')

  [ 10/110] EA       → 캐시 없음 SKIP
  [ 30/110] (패널 스킵 중... WMT)
  [ 60/110] (패널 스킵 중... ABBV)
  [ 90/110] (패널 스킵 중... SW)
  [110/110] (패널 스킵 중... EXC)

완료: 성공 109개 / 실패 1개
실패 종목: ['EA']


In [8]:
# 산출 요약 — 첫 번째 성공 티커를 샘플로 출력
panel_files = sorted(PANELS_DIR.glob('*.csv'))
print(f'=== 패널 CSV 생성 현황 ===')
print(f'생성된 파일 수: {len(panel_files)} / {len(tickers)}')

if panel_files:
    sample_path = panel_files[0]
    df_sample = pd.read_csv(sample_path, index_col='date', parse_dates=True)
    print(f'\n--- 샘플: {sample_path.stem} ---')
    print(f'행 수  : {len(df_sample)}')
    print(f'기간   : {df_sample.index[0].date()} ~ {df_sample.index[-1].date()}')
    print(f'컬럼수 : {len(df_sample.columns)}')
    print(f'컬럼   : {df_sample.columns.tolist()}')
    print(f'\nNaN 수 (컬럼별):')
    print(df_sample.isna().sum().to_string())
    print(f'\n처음 3행:')
    display(df_sample.head(3))

=== 패널 CSV 생성 현황 ===
생성된 파일 수: 110 / 110

--- 샘플: AAPL ---
행 수  : 4023
기간   : 2010-01-04 ~ 2025-12-30
컬럼수 : 28
컬럼   : ['ticker', 'gics_sector', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'dividends', 'split_ratio', 'log_return_1d', 'simple_return_1d', 'market_cap', 'mkt_rf', 'smb', 'hml', 'rmw', 'cma', 'rf', 'mom_factor', 'mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'mom_12m_skip_1m', 'vol_20d_ann', 'vol_60d_ann', 'vol_252d_ann']

NaN 수 (컬럼별):
ticker                0
gics_sector           0
open                  0
high                  0
low                   0
close                 0
adj_close             0
volume                0
dividends             0
split_ratio           0
log_return_1d         1
simple_return_1d      1
market_cap            0
mkt_rf                0
smb                   0
hml                   0
rmw                   0
cma                   0
rf                    0
mom_factor            0
mom_1m               21
mom_3m               63
mom_6m          

,ticker,gics_sector,open,high,low,close,adj_close,volume,dividends,split_ratio,...,rf,mom_factor,mom_1m,mom_3m,mom_6m,mom_12m,mom_12m_skip_1m,vol_20d_ann,vol_60d_ann,vol_252d_ann
date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,AAPL,Information Technology,7.622500,7.660714,7.585000,7.643214,6.412384,493729600,0.0,0.0,...,0.0,0.0058,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-05,AAPL,Information Technology,7.664286,7.699643,7.616071,7.656429,6.423470,601904800,0.0,0.0,...,0.0,0.0062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-06,AAPL,Information Technology,7.656429,7.686786,7.526786,7.534643,6.321295,552160000,0.0,0.0,...,0.0,-0.0005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
## 요약

- `data/panels/{ticker}.csv` : 110개 파일 (date 인덱스 + 28개 컬럼)
- `data/cache/{ticker}.pkl`   : 원본 가격 보존 (재처리 가능)
- `data/ff_factors.csv`       : 일간 FF5 + MOM 팩터

**다음 단계 (향후 세션)**
1. 품질 검증 리포트 + 기초 시계열 분석
2. 공분산 행렬 추정 (Sample / Ledoit-Wolf / EWMA)
3. Black-Litterman 입력 구성 (π, Σ, Views, Ω)
4. Walk-forward 백테스트 설계 (IS/OOS 윈도우 확정)